# 03 — Modelos predictivos de consumo eléctrico

**TFM · Sistema Inteligente de Gestión Energética** · Álvaro Bermejo Urgel

Comparativa de modelos para la predicción del consumo horario a **48 horas vista**, que es el horizonte
comprometido en el anteproyecto y el que necesita el módulo de optimización de la fase 7.

## Protocolo de evaluación

Todos los modelos se evalúan con **backtesting de origen móvil** (*rolling origin*), que reproduce la
operación real del sistema: cada día se reentrena con todo lo observado hasta ese momento y se predicen
las siguientes 48 horas. Es un criterio más exigente y más honesto que un único *train/test split*,
porque promedia el error sobre 30 puntos de partida distintos y nunca usa información futura.

## Modelos comparados

| Familia | Modelo | Justificación |
|---|---|---|
| Baseline | Naïve persistente | Suelo absoluto: repite el último valor |
| Baseline | Media móvil 24 h | Suelo con suavizado |
| Baseline | Naïve estacional 24 h | Reproduce el perfil del día anterior |
| Baseline | Naïve estacional 168 h | Capta el efecto laborable/fin de semana |
| Baseline | Media perfil semanal | Media histórica por (día, hora) |
| Estadístico | SARIMAX + Fourier (univariante) | Doble estacionalidad 24 h y 168 h |
| Estadístico | SARIMAX + Fourier + exógenas | Añade temperatura AEMET y ocupación |
| Machine learning | Gradient boosting | Relaciones no lineales entre exógenas |

> Prophet y LSTM quedaron fuera del alcance final por restricción de calendario; la justificación
> metodológica se recoge en la memoria técnica.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from tfm_energia.config import PROCESSED_DIR, SEDES

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 110

SEDE = "madrid"
print(f"Sede analizada: {SEDES[SEDE]['nombre']}")

## 1. Serie objetivo

Antes de modelar, se inspecciona el perfil que los modelos deben reproducir.

In [ ]:
df = pd.read_parquet(PROCESSED_DIR / f"enriquecido_{SEDE}.parquet").set_index("timestamp").sort_index()
y = df["consumo_total_kwh"]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

lab = df[df.index.dayofweek < 5]
for mes, etiqueta in [(1, "Enero (invierno)"), (7, "Julio (verano)")]:
    perfil = lab[lab.index.month == mes].groupby(lab[lab.index.month == mes].index.hour)[
        "consumo_total_kwh"
    ].mean()
    axes[0].plot(perfil.index, perfil.values, marker="o", ms=3, label=etiqueta)
axes[0].set(title="Perfil horario medio (laborables)", xlabel="Hora", ylabel="kWh")
axes[0].legend()

y.resample("D").sum().plot(ax=axes[1], lw=0.8)
axes[1].set(title="Consumo diario — 2 años", xlabel="", ylabel="kWh/día")
plt.tight_layout()

Se aprecian las dos estacionalidades que condicionan la elección de modelo: el **ciclo diario**
(punta matinal de puesta en régimen del HVAC) y el **ciclo semanal** (caída de fin de semana),
además de la estacionalidad anual con dos máximos, invierno y verano.

## 2. Comparativa de modelos

Resultados producidos por `scripts/train_predictivo.py`:

```bash
python scripts/train_predictivo.py --sede todas --n-origenes 30
```

In [ ]:
metricas = pd.read_csv(PROCESSED_DIR / f"metricas_modelos_{SEDE}.csv")
metricas.style.format(
    {"MAE": "{:.2f}", "RMSE": "{:.2f}", "MAPE": "{:.1f}%", "sMAPE": "{:.1f}%", "R2": "{:.3f}", "MBE": "{:.2f}"}
).background_gradient(subset=["MAE"], cmap="RdYlGn_r")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
orden = metricas.sort_values("MAE")["modelo"]

sns.barplot(data=metricas, y="modelo", x="MAE", order=orden, ax=axes[0])
axes[0].set(title="MAE por modelo (menor es mejor)", xlabel="kWh", ylabel="")

sns.barplot(data=metricas, y="modelo", x="R2", order=orden, ax=axes[1])
axes[1].set(title="R² por modelo (mayor es mejor)", xlabel="", ylabel="")
plt.tight_layout()

### Lectura de los resultados

1. **El gradient boosting gana en todas las métricas.** Aprovecha las exógenas reales (temperatura AEMET,
   ocupación, radiación) y las relaciones no lineales que los modelos lineales no capturan.
2. **El naïve estacional de 168 h es un baseline muy duro en MAE**, porque el consumo de oficina es
   fuertemente repetitivo semana a semana. Cualquier modelo que no lo bata no aporta valor.
3. **SARIMAX pierde en MAE frente al naïve-168 h pero gana en RMSE y R².** Es un resultado interpretable:
   el naïve acierta las horas planas (noche, fin de semana) y falla en los picos; SARIMAX reparte mejor el
   error en las puntas. Para el módulo de optimización, que trabaja precisamente sobre las horas de punta,
   importa más el RMSE.
4. El **MAPE es engañoso en esta serie**: de noche el consumo baja a la carga base (~5 kWh) y errores
   absolutos pequeños producen porcentajes enormes. Por eso se reporta también el sMAPE y se razona
   principalmente sobre MAE y RMSE.

## 3. Degradación del error con el horizonte

¿Hasta qué hora es fiable la predicción? Es la pregunta que determina con cuánta antelación puede
programar el sistema la climatización.

In [ ]:
horiz = pd.read_csv(PROCESSED_DIR / f"metricas_horizonte_{SEDE}.csv")
destacados = ["gradient_boosting", "sarimax_fourier_exog", "naive_estacional_168h"]

fig, ax = plt.subplots()
for modelo in destacados:
    sub = horiz[horiz["modelo"] == modelo]
    ax.plot(sub["h"], sub["MAE"], label=modelo, lw=1.8)
ax.set(title="Degradación del MAE según el horizonte", xlabel="Horas por delante (h)", ylabel="MAE (kWh)")
ax.legend()
plt.tight_layout()

## 4. Predicción frente a realidad

Un episodio concreto del backtest permite ver *cómo* falla cada modelo, no solo cuánto.

In [ ]:
bt = pd.read_parquet(PROCESSED_DIR / f"backtest_{SEDE}.parquet")
origen = sorted(bt["origen"].unique())[-3]
episodio = bt[bt["origen"] == origen]

fig, ax = plt.subplots(figsize=(13, 4.5))
real = episodio[episodio["modelo"] == destacados[0]]
ax.plot(real["timestamp"], real["real"], color="black", lw=2.2, label="Real")
for modelo in destacados:
    sub = episodio[episodio["modelo"] == modelo]
    ax.plot(sub["timestamp"], sub["pred"], lw=1.4, ls="--", label=modelo)
ax.set(title=f"Predicción a 48 h desde el origen {origen}", xlabel="", ylabel="kWh")
ax.legend()
plt.tight_layout()

## 5. Qué variables aportan de verdad

Importancia por permutación del modelo ganador: mide cuánto empeora el MAE al barajar cada variable.
Es el análisis que justifica el esfuerzo de integrar las APIs reales de AEMET y e·sios.

In [ ]:
from tfm_energia.models.ml_model import EXOGENAS_ML_DEFAULT, GradientBoostingForecaster

X = df[list(EXOGENAS_ML_DEFAULT)].astype(float)
corte = len(y) - 24 * 30

modelo = GradientBoostingForecaster(horizonte=48).fit(y.iloc[:corte], X.iloc[:corte])
importancia = modelo.importancia_permutacion(y.iloc[corte:], X.iloc[corte:], n_repeats=5)

fig, ax = plt.subplots(figsize=(10, 6))
top = importancia.head(15)
sns.barplot(data=top, y="feature", x="importancia", ax=ax)
ax.set(title="Importancia por permutación (top 15)", xlabel="Aumento del MAE al permutar", ylabel="")
plt.tight_layout()
top

## 6. Conclusiones de la fase 3

- El modelo seleccionado para el sistema es el **gradient boosting** sobre features conocidas a 48 h vista.
- Se adopta el **naïve estacional de 168 h como baseline de referencia** en la memoria: es el criterio
  honesto contra el que medir la aportación real del modelado.
- El enfoque **directo multi-paso** (lags ≥ 48 h) evita la acumulación de error de los métodos recursivos
  y garantiza que ninguna variable usada esté indisponible en el momento de predecir.
- Las predicciones alimentan la **fase 7 (optimización con PuLP)**: sobre la curva prevista a 48 h se
  decide qué fracción de la demanda de climatización puede adelantarse a horas valle sin salir de la
  banda de confort.